**Daniel Lyu, Ariel Pan**

Spring 2026

CS 443: Bio-inspired Machine Learning

# Project 4: Recurrent Neural Networks

The task in this notebook focuses on the implementation and testing of the Gated Recurrent Unit (GRU) RNN layer.

In [3]:
import tensorflow as tf
import matplotlib.pyplot as plt

from text_dataset_char import CharLevelDataset

plt.show()
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'font.size': 20})

# Automatically reload external modules
%load_ext autoreload
%autoreload 2

I0000 00:00:1778201932.704794     786 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778201932.718816     786 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778201932.862367     786 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1778201944.097668     786 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778201944.099670     786 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Task 2: Build the Gated Recurrent Unit (GRU) layer

In [4]:
from text_dataset_char import CharLevelDataset

### 2a. Load in and preprocess dev corpus

We will use the classic The Three Little Pigs nursery rhyme (`little_pigs.csv`) for our RNN dev set.

In the cell below, load in the Three Little Pigs corpus and preprocess it using a sequence length to `103` with an overlap of `10` chars between sequences.

Additionally:
- Assign the vocab to the variable `dev_vocab`.
- Assign the str coded training x and y sequences to the variables: `dev_seqs_x_str`, `dev_seqs_y_str`.
- Assign the int coded training x and y sequences to the variables: `dev_seqs_x_int`, `dev_seqs_y_int`.

In [5]:
# YOUR CODE HERE
# 1. Initialize the dataset object with the file path
dev_dataset = CharLevelDataset(file_path='data/little_pigs.csv')

# 2. Preprocess the dataset:
# N_reviews=-1: load all text
# seq_len=103: length of each sequence
# seq_overlap=10: number of characters that carry over between adjacent windows
# prop_val=0.0: ensure all data is assigned to the training split
dev_dataset.process(N_reviews=-1, seq_len=103, seq_overlap=10, prop_val=0.0)

# 3. Assign the vocabulary
dev_vocab = dev_dataset.get_vocab()

# 4. Assign the string-coded sequences
dev_seqs_x_str, dev_seqs_y_str = dev_dataset.get_train_split_str()

# 5. Assign the integer-coded (TensorFlow tensor) sequences
dev_seqs_x_int, dev_seqs_y_int = dev_dataset.get_train_split_int()

Number of unique chars/tokens: 39
Number of train sequences: 14 of length 103
  seqs_x_int.shape=TensorShape([14, 103]) seqs_y_int.shape=TensorShape([14, 103])
Number of val sequences: 0 of length 103
  seqs_x_int.shape=TensorShape([0]) seqs_y_int.shape=TensorShape([0])


E0000 00:00:1778202031.087853     786 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Let's check to make sure the dev set shapes match expectations:

In [6]:
print(f'Your vocab has {len(dev_vocab)} tokens and it should have 39.')

print('-------------TRAINING SET-------------')
print()
print('Length of your str coded sequences:')
print(f'{len(dev_seqs_x_str)},  {len(dev_seqs_y_str)}')
print('They should be:')
print('14,  14')
print()
print('Shape of your int coded sequences:')
print(f'{dev_seqs_x_int.shape}, {dev_seqs_y_int.shape}')
print('They should be:')
print('14, 103), (14, 103)')

Your vocab has 39 tokens and it should have 39.
-------------TRAINING SET-------------

Length of your str coded sequences:
14,  14
They should be:
14,  14

Shape of your int coded sequences:
(14, 103), (14, 103)
They should be:
14, 103), (14, 103)


The following cell prints out your Three Little Pig string coded sequences.

In [7]:
from text_util import decode_special_tokens

In [8]:
for chunk in dev_seqs_x_str:
    chunk = decode_special_tokens(chunk)
    print(chunk)

<START>Once upon a time there were three little pigs. The first little pig built a house of straw. The second
The second little pig built a house of sticks. The third little pig built a house of bricks. Along came
Along came a big bad wolf. He went to the house of straw and shouted: 'Little pig, little pig, let me c
, let me come in!' To which the pig replied: 'No, no, by the hair of my chinny chin chin!' So the wolf 
 the wolf said: 'Then I will huff, and I will puff, and I will blow your house down!' And he blew the h
blew the house of straw away. The first pig ran to the second pig's house of sticks. The wolf followed 
 followed and shouted: 'Little pigs, little pigs, let me come in!' 'No, no, by the hair of our chinny c
r chinny chin chins!' 'Then I will huff, and I will puff, and I will blow your house down!' And he blew
nd he blew the house of sticks away. The two pigs ran to the third pig's house of bricks. The wolf foll
 wolf followed and shouted: 'Little pigs, little pigs, let

The above cell should output:

```
<START>Once upon a time there were three little pigs. The first little pig built a house of straw. The second
The second little pig built a house of sticks. The third little pig built a house of bricks. Along came
Along came a big bad wolf. He went to the house of straw and shouted: 'Little pig, little pig, let me c
, let me come in!' To which the pig replied: 'No, no, by the hair of my chinny chin chin!' So the wolf 
 the wolf said: 'Then I will huff, and I will puff, and I will blow your house down!' And he blew the h
blew the house of straw away. The first pig ran to the second pig's house of sticks. The wolf followed 
 followed and shouted: 'Little pigs, little pigs, let me come in!' 'No, no, by the hair of our chinny c
r chinny chin chins!' 'Then I will huff, and I will puff, and I will blow your house down!' And he blew
nd he blew the house of sticks away. The two pigs ran to the third pig's house of bricks. The wolf foll
 wolf followed and shouted: 'Little pigs, little pigs, let me come in!' 'No, no, by the hair of our chi
of our chinny chin chins!' 'Then I will huff, and I will puff, and I will blow your house down!' He huf
n!' He huffed and he puffed, but he could not blow the brick house down. The wolf tried to climb down t
imb down the chimney, but he fell into a pot of boiling water and ran away forever. The three little pi
 little pigs lived happily ever after in the house of bricks.<END><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
```

The following cell prints out the first 24 tokens of each of your Three Little Pigs int coded sequences.

In [9]:
for chunk in dev_seqs_x_int[:, :24]:
    print(chunk.numpy())

[ 1 14 29 19 21  3 35 31 30 29  3 17  3 34 25 28 21  3 34 24 21 32 21  3]
[16 24 21  3 33 21 19 30 29 20  3 27 25 34 34 27 21  3 31 25 23  3 18 35]
[ 9 27 30 29 23  3 19 17 28 21  3 17  3 18 25 23  3 18 17 20  3 37 30 27]
[ 6  3 27 21 34  3 28 21  3 19 30 28 21  3 25 29  4  5  3 16 30  3 37 24]
[ 3 34 24 21  3 37 30 27 22  3 33 17 25 20  8  3  5 16 24 21 29  3 11  3]
[18 27 21 37  3 34 24 21  3 24 30 35 33 21  3 30 22  3 33 34 32 17 37  3]
[ 3 22 30 27 27 30 37 21 20  3 17 29 20  3 33 24 30 35 34 21 20  8  3  5]
[32  3 19 24 25 29 29 38  3 19 24 25 29  3 19 24 25 29 33  4  5  3  5 16]
[29 20  3 24 21  3 18 27 21 37  3 34 24 21  3 24 30 35 33 21  3 30 22  3]
[ 3 37 30 27 22  3 22 30 27 27 30 37 21 20  3 17 29 20  3 33 24 30 35 34]
[30 22  3 30 35 32  3 19 24 25 29 29 38  3 19 24 25 29  3 19 24 25 29 33]
[29  4  5  3 10 21  3 24 35 22 22 21 20  3 17 29 20  3 24 21  3 31 35 22]
[25 28 18  3 20 30 37 29  3 34 24 21  3 19 24 25 28 29 21 38  6  3 18 35]
[ 3 27 25 34 34 27 21  3 31 25 23 33  

The above cell should output:

```
[ 1 14 29 19 21  3 35 31 30 29  3 17  3 34 25 28 21  3 34 24 21 32 21  3]
[16 24 21  3 33 21 19 30 29 20  3 27 25 34 34 27 21  3 31 25 23  3 18 35]
[ 9 27 30 29 23  3 19 17 28 21  3 17  3 18 25 23  3 18 17 20  3 37 30 27]
[ 6  3 27 21 34  3 28 21  3 19 30 28 21  3 25 29  4  5  3 16 30  3 37 24]
[ 3 34 24 21  3 37 30 27 22  3 33 17 25 20  8  3  5 16 24 21 29  3 11  3]
[18 27 21 37  3 34 24 21  3 24 30 35 33 21  3 30 22  3 33 34 32 17 37  3]
[ 3 22 30 27 27 30 37 21 20  3 17 29 20  3 33 24 30 35 34 21 20  8  3  5]
[32  3 19 24 25 29 29 38  3 19 24 25 29  3 19 24 25 29 33  4  5  3  5 16]
[29 20  3 24 21  3 18 27 21 37  3 34 24 21  3 24 30 35 33 21  3 30 22  3]
[ 3 37 30 27 22  3 22 30 27 27 30 37 21 20  3 17 29 20  3 33 24 30 35 34]
[30 22  3 30 35 32  3 19 24 25 29 29 38  3 19 24 25 29  3 19 24 25 29 33]
[29  4  5  3 10 21  3 24 35 22 22 21 20  3 17 29 20  3 24 21  3 31 35 22]
[25 28 18  3 20 30 37 29  3 34 24 21  3 19 24 25 28 29 21 38  6  3 18 35]
[ 3 27 25 34 34 27 21  3 31 25 23 33  3 27 25 36 21 20  3 24 17 31 31 25]
```

### 2b. Test `Embedding` layer weights and bias

The RNN will make use of your `Embedding` layer from the SOM project. Let's test it with the Three Little Pigs dev set corpus to make sure it handles the input sequences.

In [30]:
from skipgram_layers import Embedding

In [12]:
tf.random.set_seed(0)

# TODO: Fill in appropriate units M and T below
M = len(dev_vocab)
T = 103
embed_dim = 8
embed_layer = Embedding(name='TestEmbed', units=embed_dim)
embed_layer(tf.random.uniform(shape=(1, T, M), maxval=1))
print(f'Your Embedding layer wts shape is {embed_layer.get_wts().shape} and it should be (39, 8).')
print(f'Your Embedding layer bias shape is {embed_layer.get_b().shape} and it should be (8,).')

Your Embedding layer wts shape is (39, 8) and it should be (39, 8).
Your Embedding layer bias shape is (8,) and it should be (8,).


### 2c. Questions

**Question 1:** Interpret the meaning of the embedding layer weights and bias shapes. What do the numbers mean?

**Answer 1:** YOUR ANSWER HERE
The 39 means the size of our vocabulary, one unique row for each character  in dev_vocab.
The 8 represents how many features each character have, or the "context" that we are trying to understand it in. 

### 2d. Test `Embedding layer` activations

The tests in this subtask test your `Embedding` layer activations with the dev set.

In [13]:
test_net_acts = embed_layer(dev_seqs_x_int[:3])
# print(f'Your Embedding layer wts shape is {embed_layer.get_wts().shape} and it should be (39, 8).')
# print(f'Your Embedding layer bias shape is {embed_layer.get_b().shape} and it should be (8,).')
print(f'The shape of your Embedding layer net_acts for the 1st 3 dev seqs is\n{test_net_acts.shape}.')
print('It should be\n(3, 103, 8).')
print(f"The dtype of your Embedding layer net_acts is {test_net_acts.dtype} and it should be <dtype: 'float32'>.")

The shape of your Embedding layer net_acts for the 1st 3 dev seqs is
(3, 103, 8).
It should be
(3, 103, 8).
The dtype of your Embedding layer net_acts is <dtype: 'float32'> and it should be <dtype: 'float32'>.


In [17]:
print('net acts at 1st time steps:')
print(test_net_acts[:, 0].numpy())
print('They should be:')
print('''[[ 0.12856683 -0.14906389 -0.05963672 -0.05139697 -0.44224057  0.35550532
  -0.08391167  0.3273388 ]
 [ 0.05682836  0.06774091 -0.2036617   0.18991658  0.09862589 -0.12203483
  -0.12675045 -0.04713273]
 [ 0.08161843  0.15021352 -0.24173471  0.04007699 -0.16950662  0.00373191
   0.11849379 -0.02096194]]''')
print()
print('net acts at 2nd time step:')
print(test_net_acts[:, 1].numpy())
print('They should be:')
print('''[[ 0.07443557  0.03303872 -0.22183104 -0.2542691   0.05263455  0.04380978
  -0.18369779  0.20441233]
 [-0.15169476 -0.16348508  0.00741124  0.19319822 -0.1742482   0.41682756
  -0.0234661   0.01502939]
 [-0.1909659  -0.00559532 -0.18755613 -0.19685164 -0.02494678  0.10785877
   0.2914094  -0.12029397]]''')

net acts at 1st time steps:
[[ 0.12856685 -0.1490639  -0.05963673 -0.05139695 -0.4422405   0.35550538
  -0.08391172  0.3273388 ]
 [ 0.05682835  0.0677409  -0.20366171  0.18991657  0.09862585 -0.12203483
  -0.12675045 -0.04713271]
 [ 0.08161843  0.15021352 -0.24173471  0.04007699 -0.16950664  0.00373191
   0.11849378 -0.02096194]]
They should be:
[[ 0.12856683 -0.14906389 -0.05963672 -0.05139697 -0.44224057  0.35550532
  -0.08391167  0.3273388 ]
 [ 0.05682836  0.06774091 -0.2036617   0.18991658  0.09862589 -0.12203483
  -0.12675045 -0.04713273]
 [ 0.08161843  0.15021352 -0.24173471  0.04007699 -0.16950662  0.00373191
   0.11849379 -0.02096194]]

net acts at 2nd time step:
[[ 0.07443557  0.03303872 -0.22183102 -0.2542691   0.05263454  0.04380976
  -0.1836978   0.2044123 ]
 [-0.15169476 -0.16348508  0.00741124  0.19319822 -0.17424822  0.41682756
  -0.02346611  0.0150294 ]
 [-0.1909659  -0.0055953  -0.18755613 -0.19685164 -0.02494678  0.10785878
   0.2914094  -0.12029397]]
They should be:


### 2e. Implement `GRU` constructor and weight initialization methods

The `GRU` class is located in `rnn_layers.py`. The methods to implement for the tests in this section are:
1. Constructor
2. `has_wts(self)`
3. `init_params(self, input_shape)`

For `init_params`, remember that there are 3 sets of weights/biases for:
1. Update gate
2. Reset gate
3. Candidate activation

Each item above requires 3 sets of parameters:
1. Input -> Curr Layer (feedforward) weights
2. Curr Layer -> Curr layer hidden-to-hidden unit (recurrent) weights
3. Bias

#### GRU weight initialization

Use He/Kaiming initialization for all weights/biases. For the bias and hidden-to-hidden recurrent weights, note that `fan_in` becomes the number of neurons in the *current* layer (`H`).

Recall:
- the update and reset gate weights use sigmoid activation, so the appropriate Kaiming Gain is `1.0`.
- the candidate activation uses tanh activation, so the appropriate Kaiming Gain is $\frac{5^2}{3^2}$. So if you nix the square root in the numerator of the He/Kaiming $\sigma$ calculation, the factor to multiply by is `5.0/3.0`.
   

In [31]:
from rnn_layers import GRU

#### Test: GRU weights and biases

In [33]:
tf.random.set_seed(0)
test_x = tf.ones([1, 2, 9])  # (B, T, H_prev)
test_gru_layer = GRU('TestGRULayer', units=4)  # H
test_gru_layer.init_params(input_shape=test_x.shape)

print('Update Gate test...')
test_wts = test_gru_layer.get_wts()[:2]
i2h, h2h = test_wts
b = test_gru_layer.get_b()[0]

print(f'Your GRU update i2h wts have shape={i2h.shape} and they should be (9, 4)')
print(f'Your GRU update i2h wts are a tf Variable (as they should be)? {isinstance(i2h, tf.Variable)}')
print(f'Your first few GRU update i2h wts are\n{i2h[:3]} and they should be:')
print('''[[ 0.5036876   0.14097404 -0.13989834 -0.34534574]
 [-0.412276    0.1567577  -0.00465827  0.3962861 ]
 [ 0.20084444  0.19990371 -0.23523727 -0.1443252 ]]''')

print(f'Your GRU update h2h wts have shape={h2h.shape} and they should be (4, 4)')
print(f'Your GRU update h2h wts are a tf Variable (as they should be)? {isinstance(h2h, tf.Variable)}')
print(f'Your first few GRU update h2h wts are\n{h2h[:3]} and they should be:')
print('''[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]''')

print(f'Your GRU update bias have shape={b.shape} and they should be (4,)')
print(f'Your GRU update bias are a tf Variable (as they should be)? {isinstance(b, tf.Variable)}')
print(f'Your first few GRU update bias are\n{b[:3]} and they should be:')
print('''[0. 0. 0.]''')

aaaa
Update Gate test...
Your GRU update i2h wts have shape=(9, 4) and they should be (9, 4)
Your GRU update i2h wts are a tf Variable (as they should be)? True
Your first few GRU update i2h wts are
[[ 0.50368756  0.14097401 -0.13989832 -0.34534574]
 [-0.41227597  0.15675768 -0.0046583   0.3962861 ]
 [ 0.20084445  0.19990371 -0.2352373  -0.14432515]] and they should be:
[[ 0.5036876   0.14097404 -0.13989834 -0.34534574]
 [-0.412276    0.1567577  -0.00465827  0.3962861 ]
 [ 0.20084444  0.19990371 -0.23523727 -0.1443252 ]]
Your GRU update h2h wts have shape=(4, 4) and they should be (4, 4)
Your GRU update h2h wts are a tf Variable (as they should be)? True
Your first few GRU update h2h wts are
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]] and they should be:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
Your GRU update bias have shape=(4,) and they should be (4,)
Your GRU update bias are a tf Variable (as they should be)? True
Your first few GRU update bias are
[0. 0. 0.] and they should

In [34]:
print('Reset Gate test...')
test_wts = test_gru_layer.get_wts()[2:4]
i2h, h2h = test_wts
b = test_gru_layer.get_b()[1]

print(f'Your GRU reset i2h wts have shape={i2h.shape} and they should be (9, 4)')
print(f'Your GRU reset i2h wts are a tf Variable (as they should be)? {isinstance(i2h, tf.Variable)}')
print(f'Your GRU reset i2h wts are\n{i2h[:3]} and they should be:')
print('''[[ 0.35562676  0.06484976 -0.176943    0.03063358]
 [-0.05917902 -0.3064363  -0.69258523  0.6797302 ]
 [ 0.2676332  -0.3103012  -0.12414372 -0.10699131]]''')

print(f'Your GRU reset h2h wts have shape={h2h.shape} and they should be (4, 4)')
print(f'Your GRU reset h2h wts are a tf Variable (as they should be)? {isinstance(h2h, tf.Variable)}')
print(f'Your first few GRU reset h2h wts are\n{h2h[:3]} and they should be:')
print('''[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]''')

print(f'Your GRU reset bias have shape={b.shape} and they should be (4,)')
print(f'Your GRU reset bias are a tf Variable (as they should be)? {isinstance(b, tf.Variable)}')
print(f'Your first few GRU reset bias are\n{b[:3]} and they should be:')
print('''[0. 0. 0.]''')

Reset Gate test...
Your GRU reset i2h wts have shape=(9, 4) and they should be (9, 4)
Your GRU reset i2h wts are a tf Variable (as they should be)? True
Your GRU reset i2h wts are
[[ 0.35562676  0.06484976 -0.17694299  0.03063361]
 [-0.05917903 -0.3064363  -0.69258523  0.6797301 ]
 [ 0.26763323 -0.31030124 -0.12414374 -0.10699128]] and they should be:
[[ 0.35562676  0.06484976 -0.176943    0.03063358]
 [-0.05917902 -0.3064363  -0.69258523  0.6797302 ]
 [ 0.2676332  -0.3103012  -0.12414372 -0.10699131]]
Your GRU reset h2h wts have shape=(4, 4) and they should be (4, 4)
Your GRU reset h2h wts are a tf Variable (as they should be)? True
Your first few GRU reset h2h wts are
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]] and they should be:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
Your GRU reset bias have shape=(4,) and they should be (4,)
Your GRU reset bias are a tf Variable (as they should be)? True
Your first few GRU reset bias are
[0. 0. 0.] and they should be:
[0. 0. 0.]


In [35]:
print('Candidate activation test...')
test_wts = test_gru_layer.get_wts()[4:]
i2h, h2h = test_wts
b = test_gru_layer.get_b()[-1]

print(f'Your GRU candidate i2h wts have shape={i2h.shape} and they should be (9, 4)')
print(f'Your GRU candidate i2h wts are a tf Variable (as they should be)? {isinstance(i2h, tf.Variable)}')
print(f'Your GRU candidate i2h wts are\n{i2h[:3]} and they should be:')
print('''[[-1.0022936  -0.06196385 -0.4697506   0.4716453 ]
 [ 0.06798682  0.741693    0.3442466  -0.30546117]
 [-0.33790386  0.47027883 -0.31180242 -1.0740486 ]]''')

print(f'Your GRU candidate h2h wts have shape={h2h.shape} and they should be (4, 4)')
print(f'Your GRU candidate h2h wts are a tf Variable (as they should be)? {isinstance(h2h, tf.Variable)}')
print(f'Your first few GRU candidate h2h wts are\n{h2h[:3]} and they should be:')
print('''[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]''')

print(f'Your GRU candidate bias have shape={b.shape} and they should be (4,)')
print(f'Your GRU candidate bias are a tf Variable (as they should be)? {isinstance(b, tf.Variable)}')
print(f'Your first few GRU candidate bias are\n{b[:3]} and they should be:')
print('''[0. 0. 0.]''')

Candidate activation test...
Your GRU candidate i2h wts have shape=(9, 4) and they should be (9, 4)
Your GRU candidate i2h wts are a tf Variable (as they should be)? True
Your GRU candidate i2h wts are
[[-1.0022936  -0.06196362 -0.46975052  0.4716452 ]
 [ 0.06798682  0.74169296  0.34424663 -0.30546114]
 [-0.3379038   0.47027892 -0.3118025  -1.0740486 ]] and they should be:
[[-1.0022936  -0.06196385 -0.4697506   0.4716453 ]
 [ 0.06798682  0.741693    0.3442466  -0.30546117]
 [-0.33790386  0.47027883 -0.31180242 -1.0740486 ]]
Your GRU candidate h2h wts have shape=(4, 4) and they should be (4, 4)
Your GRU candidate h2h wts are a tf Variable (as they should be)? True
Your first few GRU candidate h2h wts are
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]] and they should be:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
Your GRU candidate bias have shape=(4,) and they should be (4,)
Your GRU candidate bias are a tf Variable (as they should be)? True
Your first few GRU candidate bias are
[0. 0

### 2f. Implement `GRU` `net_in` and `net_act`

Because there is no "unitary" net input and activation calculation in a GRU layer, we have to calculate netIn and netActs for all 3 GRU components in each method:
- update gates
- reset gates
- candidate activation

#### Test: `compute_net_input`

In [39]:
test_x = tf.ones([3, 9])  # (B, H_prev)

tf.random.set_seed(0)
test_prev_state = tf.random.uniform(shape=(3, 4))  # (B, T, H)

u, r, c = test_gru_layer.compute_net_input(test_x, test_prev_state)
print(f'Your update gate net_in shape is {u.shape} and it should be (3, 4).')
print(f'Your update gate net_in 1st few values are\n{tf.reshape(u, -1)[:6]}')
print('and they should be')
print('[ 0.21471995  1.071414   -0.304483    0.44639048  0.3394193   1.6726756 ]')
print(f'Your reset gate net_in shape is {r.shape} and it should be (3, 4).')
print(f'Your reset gate net_in 1st few values are\n{tf.reshape(r, -1)[:6]}')
print('and they should be')
print('[-0.8721906   0.15900365 -1.2871578   1.8869963  -0.74749124  0.7602651 ]')
print(f'Your candidate net_in shape is {c.shape} and it should be (3, 4).')
print(f'Your candidate net_in 1st few values are\n{tf.reshape(c, -1)[:6]}')
print('and they should be')
print('[-0.75695324 -0.4416539  -2.0830038  -0.60320866 -0.75695324 -0.4416539 ]')


Your update gate net_in shape is (3, 4) and it should be (3, 4).
Your update gate net_in 1st few values are
[ 0.21471983  1.071414   -0.30448306  0.44639045  0.3394192   1.6726754 ]
and they should be
[ 0.21471995  1.071414   -0.304483    0.44639048  0.3394193   1.6726756 ]
Your reset gate net_in shape is (3, 4) and it should be (3, 4).
Your reset gate net_in 1st few values are
[-0.8721907   0.15900385 -1.2871578   1.8869963  -0.74749136  0.76026535]
and they should be
[-0.8721906   0.15900365 -1.2871578   1.8869963  -0.74749124  0.7602651 ]
Your candidate net_in shape is (3, 4) and it should be (3, 4).
Your candidate net_in 1st few values are
[-0.7569531 -0.4416538 -2.0830033 -0.6032082 -0.7569531 -0.4416538]
and they should be
[-0.75695324 -0.4416539  -2.0830038  -0.60320866 -0.75695324 -0.4416539 ]


#### Test: `net_act`

In [43]:
act, ua, ra = test_gru_layer.compute_net_activation(u, r, c, test_prev_state)

print(f'Your net_act shape is {act.shape} and it should be (3, 4).')
print(f'Your net_act 1st few values are\n{tf.reshape(act, -1)[:6]}')
print('and they should be')
print('[-0.1937173  -0.18467134 -0.10003933  0.1487115  -0.14980711  0.21894169]')
print(f'Your update gate net_act shape is {ua.shape} and it should be (3, 4).')
print(f'Your update gate net_act 1st few values are\n{tf.reshape(ua, -1)[:6]}')
print('and they should be')
print('[0.5534747  0.7448657  0.42446196 0.6097807  0.58404946 0.84193224]')
print(f'Your reset net_act shape is {ra.shape} and it should be (3, 4).')
print(f'Your reset net_act 1st few values are\n{tf.reshape(ra, -1)[:6]}')
print('and they should be')
print('[0.29479867 0.53966737 0.21633427 0.86841273 0.3213682  0.6814113 ]')

Your net_act shape is (3, 4) and it should be (3, 4).
Your net_act 1st few values are
[-0.19371712 -0.1846712  -0.10003909  0.14871182 -0.14980704  0.21894187]
and they should be
[-0.1937173  -0.18467134 -0.10003933  0.1487115  -0.14980711  0.21894169]
Your update gate net_act shape is (3, 4) and it should be (3, 4).
Your update gate net_act 1st few values are
[0.5534746  0.74486566 0.42446187 0.60978067 0.5840494  0.8419322 ]
and they should be
[0.5534747  0.7448657  0.42446196 0.6097807  0.58404946 0.84193224]
Your reset net_act shape is (3, 4) and it should be (3, 4).
Your reset net_act 1st few values are
[0.29479867 0.53966737 0.21633428 0.8684127  0.3213682  0.6814114 ]
and they should be
[0.29479867 0.53966737 0.21633427 0.86841273 0.3213682  0.6814113 ]


### 2g. The forward pass through the `GRU` layer 

This involves implementing the following methods in `rnn_layers.py`:
1. `reset_state(self, B)`
2. `__call__(self, x, mask, state=None)`

Remember that the forward pass involves computes netIn/netAct for each time step **sequentially** (i.e. unroll across time) and update the state in a GRU layer based on the update gates, reset gates, and candidate activations. The forward pass method returns a history/record of the GRU evolving state at each time step in the mini-batch.

**Note:**
- `__call__` takes in as input the padding mask for the current mini-batch, which contains 1 for any time step of any sequence where the current token is the <PAD> token, otherwise 0. The idea is we don't want to train the net to predict padding tokens, so we don't allow the padding token to change/update GRU states (*otherwise that would wipe out medium-term memories of USEFUL chars*).
- `__call__` also optionally takes in a prior state. During training, we won't use this (i.e. each mini-batch is processed from a reset initial state). However, during inference/text generation, we need to feed the GRU the state based on all the text it has previously generated.

#### Test: `forward`

No prior state.

In [44]:
tf.random.set_seed(0)
test_x = tf.random.uniform(shape=[3, 2, 9], maxval=1)  # (B, T, H_prev)
test_mask = tf.ones([3, 2, 1])  # (B, T, 1)
state_hist = test_gru_layer(test_x, test_mask)

print(f'After 1 time step, your netAct/state looks like:\n{tf.squeeze(state_hist[0])}')
print('and it should be:')
print('''[[-0.02641175 -0.515999   -0.37362885 -0.02108434]
 [ 0.04090836 -0.5428403  -0.51041895 -0.22800834]]''')

print()
print(f'After 2nd time step, your netAct/state looks like:\n{tf.squeeze(state_hist[1])}')
print('and it should be:')
print('''[[-0.2985971  -0.52458036 -0.350078    0.2881527 ]
 [-0.4101661  -0.36315447 -0.45769507 -0.0883065 ]]''')

After 1 time step, your netAct/state looks like:
[[-0.0264117  -0.515999   -0.3736287  -0.02108422]
 [ 0.04090842 -0.5428403  -0.5104188  -0.22800814]]
and it should be:
[[-0.02641175 -0.515999   -0.37362885 -0.02108434]
 [ 0.04090836 -0.5428403  -0.51041895 -0.22800834]]

After 2nd time step, your netAct/state looks like:
[[-0.29859707 -0.52458036 -0.35007793  0.28815278]
 [-0.41016608 -0.3631545  -0.457695   -0.08830637]]
and it should be:
[[-0.2985971  -0.52458036 -0.350078    0.2881527 ]
 [-0.4101661  -0.36315447 -0.45769507 -0.0883065 ]]


Prior state

In [45]:
tf.random.set_seed(1)
test_prior_state = tf.random.uniform(shape=[3, 4], maxval=1)  # (B, H)
state_hist = test_gru_layer(test_x, test_mask, state=test_prior_state)

print(f'After 1 time step, your netAct/state looks like:\n{tf.squeeze(state_hist[0])}')
print('and it should be:')
print('''[[ 0.08932833 -0.17840748 -0.21178606  0.3274149 ]
 [ 0.13222653 -0.35804257 -0.42159346  0.01643313]]''')

print()
print(f'After 2nd time step, your netAct/state looks like:\n{tf.squeeze(state_hist[1])}')
print('and it should be:')
print('''[[-0.15771979 -0.33748418 -0.08159453  0.6183338 ]
 [-0.3357961  -0.24368615 -0.2849819   0.12609966]]''')

After 1 time step, your netAct/state looks like:
[[ 0.08932839 -0.17840753 -0.21178591  0.32741505]
 [ 0.13222662 -0.35804254 -0.42159337  0.01643345]]
and it should be:
[[ 0.08932833 -0.17840748 -0.21178606  0.3274149 ]
 [ 0.13222653 -0.35804257 -0.42159346  0.01643313]]

After 2nd time step, your netAct/state looks like:
[[-0.15771976 -0.33748403 -0.08159456  0.61833376]
 [-0.33579603 -0.24368608 -0.2849819   0.12609966]]
and it should be:
[[-0.15771979 -0.33748418 -0.08159453  0.6183338 ]
 [-0.3357961  -0.24368615 -0.2849819   0.12609966]]


Processing a new mini-batch without specifying a prior state so GRU should reset the state.

In [46]:
tf.random.set_seed(2)
test_x2 = tf.random.uniform(shape=[3, 2, 9], minval=1, maxval=2)  # (B, T, H_prev)
state_hist = test_gru_layer(test_x2, test_mask)

print(f'After 1 time step, your netAct/state looks like:\n{tf.squeeze(state_hist[0])}')
print('and it should be:')
print('''[[-0.37745613 -0.5211924  -0.15695949 -0.34759605]
 [-0.5151514  -0.4136088  -0.34603268 -0.56110054]]''')

print()
print(f'After 2nd time step, your netAct/state looks like:\n{tf.squeeze(state_hist[1])}')
print('and it should be:')
print('''[[-0.3615895  -0.6409677  -0.21561074 -0.29363844]
 [-0.5962447  -0.6232561  -0.34108102 -0.50182986]]''')

After 1 time step, your netAct/state looks like:
[[-0.37745604 -0.5211923  -0.15695947 -0.34759563]
 [-0.51515126 -0.41360885 -0.34603268 -0.5611003 ]]
and it should be:
[[-0.37745613 -0.5211924  -0.15695949 -0.34759605]
 [-0.5151514  -0.4136088  -0.34603268 -0.56110054]]

After 2nd time step, your netAct/state looks like:
[[-0.3615895  -0.6409677  -0.21561074 -0.29363826]
 [-0.5962447  -0.623256   -0.34108096 -0.5018295 ]]
and it should be:
[[-0.3615895  -0.6409677  -0.21561074 -0.29363844]
 [-0.5962447  -0.6232561  -0.34108102 -0.50182986]]


Now test with mask with both 0s and 1s in it.

In [47]:
tf.random.set_seed(1)
test_mask2 = tf.cast(tf.random.uniform(shape=[3, 2, 1], maxval=2, dtype=tf.int32), tf.float32)  # (B, T, 1)
print(f'Using the mask:\n{tf.squeeze(test_mask2)}')
state_hist = test_gru_layer(test_x2, test_mask2)

print(f'After 1 time step, your netAct/state looks like:\n{tf.squeeze(state_hist[0])}')
print('and it should be:')
print('''[[ 0.          0.          0.          0.        ]
 [-0.32633758 -0.10955579 -0.25291246 -0.3969905 ]]''')

print()
print(f'After 2nd time step, your netAct/state looks like:\n{tf.squeeze(state_hist[1])}')
print('and it should be:')
print('''[[-0.3615895  -0.6409677  -0.21561074 -0.29363844]
 [-0.5962447  -0.6232561  -0.34108102 -0.50182986]]''')

Using the mask:
[[0. 1.]
 [1. 1.]
 [0. 0.]]
After 1 time step, your netAct/state looks like:
[[ 0.          0.          0.          0.        ]
 [-0.3263375  -0.10955589 -0.25291237 -0.39699045]]
and it should be:
[[ 0.          0.          0.          0.        ]
 [-0.32633758 -0.10955579 -0.25291246 -0.3969905 ]]

After 2nd time step, your netAct/state looks like:
[[-0.3615895  -0.6409677  -0.21561074 -0.29363826]
 [-0.5962447  -0.623256   -0.34108096 -0.5018295 ]]
and it should be:
[[-0.3615895  -0.6409677  -0.21561074 -0.29363844]
 [-0.5962447  -0.6232561  -0.34108102 -0.50182986]]
